## Evaluation Framework

The purpose of this notebook is to evaluate the vector search component of our RAG model. In order to do so, we consider two pre-processed datasets respectively for job and skills data. The first one contains 552 job titles, descriptions and synthetic queries that a Compass user might submit to the platform, as well as a ground truth ESCO code for occupation and its related essential skills, contained as a list of Tabiya UUIDs for skills. Further information can be found in the [Hahu test Huggingface repository](https://huggingface.co/datasets/tabiya/hahu_test). The second dataset contains 1054 sentences with multiple skills (2013 in total) which are extracted from job description requirements. In this case, we also generated a synthetic query for the test input to match closer our use case. Further information can be found in the [Skill test set Huggingface repository](https://huggingface.co/datasets/tabiya/esco_skills_test). 

In order to set an evaluation target, let us consider our downstream application. In fact, the ESCO nodes found by our vector search will be used to build a prompt so that the Large Language Model can ask users to validate a number of skills and occupations found through vector search. Because of this two step process, we don't want to focus so much on the precision (that is, reducing the amount of false positives), but rather on the recall (that is, retrieving the largest amount of correct nodes). We also envision a reranking component that will be capable of highlighting the most important skills for a measure that could be defined externally (based, for instance, on the intrinsic value of various skills) or internally (if the user has mentioned related nodes multiple times).


On this premise, we focus our evaluation method on the recall@k, which is a metric that measures how many correct nodes are found within the top k classes. The precision@k, which tells us how many of the k retrieved nodes are correct, will be a secondary metric to consider to check how many false positives we retrieve, penalizing the use of larger sample sizes. The two metrics are summarized by the F-score@k. Each of these metrics could be further refined by considering the score to be inversely proportional to the rank in which the correct node is found. However, since our first iteration is not concerned with the rank, we will use a 0-1 score.

In the case of occupations, since our test set includes only one correct node per sentences, the recall@k reduces to understanding whether the correct node is within the first k elements. However, multiple skill nodes can correspond to a single sentence, so that the recall at k really captures what percentage of the correct skills on average is retrieved within the first k elements.

We start by defining our metrics.


In [1]:
from typing import Dict, List, Optional, Tuple

def precision_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average precision at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total retrieved nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of samples of the prediction to consider.
            When None considers all the elements of the list.

    Returns:
        float: average precision at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_precision = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
            tot_samples = k
        else:
            tot_samples = len(pred_list)
        total_precision+=len(set(pred_list).intersection(set(true_val)))/tot_samples
    return total_precision/len(true)

def recall_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average recall at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total correct nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        float: average recall at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_recall = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
        total_recall+=len(set(pred_list).intersection(set(true_val)))/len(set(true_val))
    return total_recall/len(true)

def f_score(prec: float, rec: float) -> float:
    """Returns the f-score corresponding to
    a given precision and recall.

    Args:
        prec (float): provided precision
        rec (float): provided recall

    Returns:
        float: resulting f-score
    """
    return 2*prec*rec/(prec+rec)

def get_all_metrics(predictions: List[List[str]], true_values: List[List[str]], k: Optional[int]=None) -> Tuple[float,float,float]:
    """Get recall, precision and F-score for given results and
    true values.

    Args:
        predictions (List[List[str]]): list of predictions.
        true_values (List[List[str]]): list of true values.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        Tuple[float,float,float]: recall, precision and F-score.
    """
    rec_at_k = recall_at_k(predictions, true_values, k)
    prec_at_k = precision_at_k(predictions, true_values, k)
    f_score_at_k = f_score(prec_at_k, rec_at_k)
    return rec_at_k, prec_at_k, f_score_at_k


## Evaluation goals

We aim to run multiple evaluations on different goals, fixing the embedding model we're using as the Vertex AI Gecko003 model. We consider other variables in our experiment setting them as hyperparameters. In particular we would like to decide:

1. Which hyperparameter we should choose to properly retrieve a node.
2. How we can retrieve skills related to a query concerning a job.
3. Whether job titles are better indicators than job descriptions when retrieving the correct information.

The corresponding test datasets will be loaded from the aforementioned Huggingface respositories, while the skill, occupation and occupation-skill relational databases will be loaded from the public Tabiya Github repository [tabiya-open-dataset](https://github.com/tabiya-tech/tabiya-open-dataset/tree/main/tabiya-esco-v1.1.1). 

The following code loads the datasets, defines a series of functions useful to all three evaluations.

In [2]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv()

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
OCCUPATION_REPO_ID = "tabiya/hahu_test"
OCCUPATION_FILENAME = "redacted_hahu_test_with_id.csv"
SKILL_REPO_ID = "tabiya/esco_skills_test"
SKILL_FILENAME = "data/processed_skill_test_set_with_id.parquet"
OCCUPATION_DATA_PATH = "https://raw.githubusercontent.com/tabiya-tech/taxonomy-model-application/main/data-sets/csv/tabiya-esco-v1.1.1/occupations.csv"
SKILL_DATA_PATH = "https://raw.githubusercontent.com/tabiya-tech/taxonomy-model-application/main/data-sets/csv/tabiya-esco-v1.1.1/skills.csv"
OCCUPATION_TO_SKILL_DATA_PATH = "https://raw.githubusercontent.com/tabiya-tech/tabiya-open-dataset/main/tabiya-esco-v1.1.1/csv/occupation_skill_relations.csv"

df_occupation_to_skills = pd.read_csv(OCCUPATION_TO_SKILL_DATA_PATH)

df_occupation_test = pd.read_csv(
    hf_hub_download(repo_id=OCCUPATION_REPO_ID, filename=OCCUPATION_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_skill_test = pd.read_parquet(
    hf_hub_download(repo_id=SKILL_REPO_ID, filename=SKILL_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_occupation_database = pd.read_csv(OCCUPATION_DATA_PATH)
df_skill_database = pd.read_csv(SKILL_DATA_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

/Users/francescopreta/miniconda3/envs/backend/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Normalize skills and occupation test target fields with the database targets
df_skill_database = df_skill_database.rename(columns={"UUIDHISTORY":"CODE"})
df_skill_test["CODE"] = df_skill_test["UUID"].apply(lambda x: [x])
df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])
df_occupation_test["skills_essential"] = df_occupation_test["skills_essential"].apply(eval)
df_occupation_test["skills_optional"] = df_occupation_test["skills_optional"].apply(eval)


In [4]:
# Functions "maximal_marginal_relevance" and "cosine_similarity"
# are duplicated respectively from modules:
#    - "libs/community/langchain_community/vectorstores/utils.py"
#    - "libs/community/langchain_community/utils/math.py"
from typing import List, Union

import numpy as np

Matrix = Union[List[List[float]], List[np.ndarray], np.ndarray]


def cosine_similarity(X: Matrix, Y: Matrix) -> np.ndarray:
    """Row-wise cosine similarity between two equal-width matrices."""
    if len(X) == 0 or len(Y) == 0:
        return np.array([])

    X = np.array(X)
    Y = np.array(Y)
    if X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Number of columns in X and Y must be the same. X has shape {X.shape} "
            f"and Y has shape {Y.shape}."
        )
    X_norm = np.linalg.norm(X, axis=1)
    Y_norm = np.linalg.norm(Y, axis=1)
    # Ignore divide by zero errors run time warnings as those are handled below.
    with np.errstate(divide="ignore", invalid="ignore"):
        similarity = np.dot(X, Y.T) / np.outer(X_norm, Y_norm)
    similarity[np.isnan(similarity) | np.isinf(similarity)] = 0.0
    return similarity



def maximal_marginal_relevance(
    query_embedding: np.ndarray,
    embedding_list: list,
    lambda_mult: float = 0.5,
    k: int = 10,
) -> List[int]:
    """Calculate maximal marginal relevance."""
    if min(k, len(embedding_list)) <= 0:
        return []
    if query_embedding.ndim == 1:
        query_embedding = np.expand_dims(query_embedding, axis=0)
    similarity_to_query = cosine_similarity(query_embedding, embedding_list)[0]
    most_similar = int(np.argmax(similarity_to_query))
    idxs = [most_similar]
    selected = np.array([embedding_list[most_similar]])
    while len(idxs) < min(k, len(embedding_list)):
        best_score = -np.inf
        idx_to_add = -1
        similarity_to_selected = cosine_similarity(embedding_list, selected)
        for i, query_score in enumerate(similarity_to_query):
            if i in idxs:
                continue
            redundant_score = max(similarity_to_selected[i])
            equation_score = (
                lambda_mult * query_score - (1 - lambda_mult) * redundant_score
            )
            if equation_score > best_score:
                best_score = equation_score
                idx_to_add = i
        idxs.append(idx_to_add)
        selected = np.append(selected, [embedding_list[idx_to_add]], axis=0)
    return idxs


In what follows, we will pre-compute the strings and the corresponding embeddings using the Gecko model. We will use manual batching to speed up the process, as the vertex API doesn't support batching and fails if the list length is larger than 250 elements or the sum of tokens is higher than 20.000.

In [5]:
def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

### 1. Hyperparameter selection

The objective of this first evaluation is to choose the hyperparameters which guarantee the highest recall at each k. We select a combination of the following hyperparameters for k equal to 1, 3, 5 and 10:

1. **How to embed a node of the graph**: which combination of the fields guarantees the best performance when embedded. We consider embedding only the **preferred label**, only the **description**, the **label and description** or the **label, description and secondary labels**.
2. **Score function**: which function should be used to retrieve the most similar nodes (*cosine*, *l2 distance* or *scalar product*).
3. **Using Maximal Marginal Relevance**: whether we should use **MMR** to get more diverse results. Maximal marginal relevance is an optimization algorithm that for a given subset of retrieved nodes chooses the best and most diverse nodes in terms of similarity with each other.

We will use ChromaDB as a local vector store and get the ESCO data from a local csv file. We will restrict our evaluation to the gecko003 model, but this can be repeated with any other model.

The first evaluation will be conducted as follows:

- Evaluation of the occupation nodes from the occupation dataset.
- Evaluation of the skill nodes from the skill dataset.

In [6]:
# Functions defining strings to embed
def description(df):
    return df["DESCRIPTION"]

def preferred_label(df):
    return df["PREFERREDLABEL"]

def all_occupations(df):
    return f"""Occupation Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Occupation Description: {df['DESCRIPTION']}"""

def all_skills(df):
    return f"""Skill Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Skill Description: {df['DESCRIPTION']}"""

def label_and_description(df):
    return f"{df['PREFERREDLABEL']}\n{df['DESCRIPTION']}"

In [7]:
# Embedding precomputation

def precompute_embeddings(df_database: pd.DataFrame, function_to_method: Dict[str,Any]) -> pd.DataFrame:
    """For a given database and map that sends each function name to
    its respective function for selecting a substring from the node,
    returns an updated dataframe with the corresponding methods as
    well as embeddings for all the methods

    Args:
        df_database (pd.DataFrame): database of interest
        function_to_method (Dict[str,Any]): map from function
            name to function selecting string for any given node.

    Returns:
        The updated dataframe with the method strings and the corresponding
        embeddings.
    """
    for method in function_to_method:
        df_database[method] = df_database.progress_apply(function_to_method[method], axis=1)
        embeddings = embed_strings_in_batch(list(df_database[method]), model)
        df_database[f'embeddings_{method}'] = embeddings
    return df_database

function_to_occupation_method = {"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_OCCUPATIONS":all_occupations, "LABEL_AND_DESCRIPTION": label_and_description}
function_to_skill_method = {"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_SKILLS":all_skills, "LABEL_AND_DESCRIPTION": label_and_description}

# Compute database embeddings
df_occupation_database = precompute_embeddings(df_occupation_database, function_to_occupation_method)
df_skill_database = precompute_embeddings(df_skill_database, function_to_skill_method)

# Compute test set embeddings
test_occupation_embeddings = embed_strings_in_batch(list(df_occupation_test["synthetic_query"]), model)
df_occupation_test["embeddings"] = test_occupation_embeddings

test_skill_embeddings = embed_strings_in_batch(list(df_skill_test["synthetic_query"]), model)
df_skill_test["embeddings"] = test_skill_embeddings

NameError: name 'Dict' is not defined

We create multiple chromadb collections to store our data in memory with different embeddings depending on the method used and on the function used for querying. On these, we save the Occupation ESCO database with all their metadatas.

In [8]:
import chromadb

SCORE_FUNCTIONS = ["cosine", "l2", "ip"]
client = chromadb.Client()

def create_collections(db_name: str, methods: List[str], df_database: pd.DataFrame):
    """Creates multiple collections for each choice of db_name
    and corresponding documents and embeddings.

    Args:
        db_name (str): name of the database. 
            Either 'skills' or 'occupations'.
        methods (List[str]): list of methods for the embeddings.
        df_database (pd.DataFrame): database containing the metadata.
    """
    for method in methods:
        for score in SCORE_FUNCTIONS:
            collection_name = f'{db_name}_{method}_{score}'
            collection_metadata = {"hnsw:space":score}
            collection = client.create_collection(name=collection_name, metadata=collection_metadata)
            collection.add(
                documents = list(df_database[method]),
                metadatas = [{"CODE": row["CODE"]} for _, row in df_database.iterrows()],
                embeddings = list(df_database[f"embeddings_{method}"]),
                ids = [f"id_{i}" for i in range(len(df_database))]
            )

In [9]:
create_collections("occupations", list(function_to_occupation_method.keys()), df_occupation_database)
create_collections("skills", list(function_to_skill_method.keys()), df_skill_database)

Finally, we write a function to run the evaluation. The one linking skills to skills and occupations to occupations works as follows:

1. We choose a score function and a method and load the corresponding collection.
2. For each element in the test set, we find the top 100 documents in the collection ordered by scoring rank.
3. We filter those documents by maximal marginal relevance to find the top 10 documents with this function.
4. We evaluate the precision, recall and F-score on the top k for k=1,3,5,10 for the original retrieved documents.
5. We evaluate the precision, recall and F-score on the top k for k=1,3,5,10 for the documents filtered by maximal marginal relevance.
6. We save the results in a dataframe to be analyzed.

In [11]:
def get_results_from_embeddings(embeddings: List[List[float]], collection: chromadb.Collection) -> Tuple[List[List[str]], List[List[str]]]:
    """Utility function to return results of embedding queries
    to a given collection.

    Args:
        embeddings (List[List[float]]): List of embeddings for queries.
        collection (chromadb.Collection): ChromaDB collection to query.

    Returns:
        Tuple[List[List[str]], List[List[str]]]: List of results, one for
            regular vector search, the other one for maximal marginal relevance
            search. Each element is a list of string corresponding to the
            search result for the embedding in the same position in the input list. 
    """
    mmr_vector_search_results = []
    vector_search_results = []
    for embedding in embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=embedding, n_results=100, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
        # Find the top 10 documents according to MMR and save them in mmr_vector_search_results
        result_embeddings = [elem for elem in documents_from_search["embeddings"][0]]
        mmr_ids = maximal_marginal_relevance(np.array(embedding), result_embeddings)
        mmr_vector_search_results.append([elem["CODE"] for index, elem in enumerate(documents_from_search["metadatas"][0]) if index in mmr_ids])
    return vector_search_results, mmr_vector_search_results


def run_eval(db_type: str, method_list: List[str], score_function_list: List[str], df_test: pd.DataFrame, target_column: str = "CODE", embedding_column = "embeddings") -> pd.DataFrame:
    """Returns the results of an evaluation on a list of collections

    Args:
        db_type (str): name of the database (occupation or skill).
        method_list (List[str]): list of methods to be tested.
        score_function_list (List[str]): list of score functions to be tested.
        df_test (pd.DataFrame): test dataframe, containing an embedding column
            and a test_target column.

    Returns:
        pd.DataFrame: dataframe with the result of the evaluation depending on the
            different hyperparameters.
    """
    eval_data = []
    for method in method_list:
        for score in score_function_list:
            collection_name = f"{db_type}_{method}_{score}"
            # Fetch collection
            collection = client.get_collection(name=collection_name)
            # Initialize lists to save results
            vector_search_results, mmr_vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection)
            # Evaluate accuracy at k for k=1, 3, 5, 10
            for k in [1, 3, 5, 10]:
                rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(vector_search_results, list(df_test[target_column]), k)
                eval_data.append({"method":method, "score function":score, "MMR": False, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
                rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(mmr_vector_search_results, list(df_test[target_column]), k)
                eval_data.append({"method":method, "score function":score, "MMR": True, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
    # Save the results in a dataframe
    eval_df = pd.DataFrame(eval_data)
    return eval_df



We can now run the evaluations for occupation and skill vector search:

In [17]:
# Initialize a folder to save the evaluation results locally
OUTPUT_EVAL_PATH = os.environ.get("OUTPUT_EVAL_PATH", "")
if OUTPUT_EVAL_PATH:
    os.makedirs(OUTPUT_EVAL_PATH, exist_ok=True)

# Evaluation of occupation
df_occupation_eval = run_eval("occupations", list(function_to_occupation_method.keys()), SCORE_FUNCTIONS, df_occupation_test)
if OUTPUT_EVAL_PATH:
    df_occupation_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "occupation_eval.csv"))

# Evaluation of skills
df_skill_eval = run_eval("skills", list(function_to_skill_method.keys()), SCORE_FUNCTIONS, df_skill_test)
if OUTPUT_EVAL_PATH:
    df_skill_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "skill_eval.csv"))

Let's now discuss the results of our experiments

### Occupations

The following table illustrates the result of our experiments:

| Method              | Score Function | MMR   | k  | Recall | Precision | F-Score |
|---------------------|----------------|-------|----|--------|-----------|---------|
| PREFERREDLABEL      | cosine         | False | 10 | 0.7454 | 0.0745    | 0.1355  |
| PREFERREDLABEL      | l2             | False | 10 | 0.7454 | 0.0745    | 0.1355  |
| PREFERREDLABEL      | ip             | False | 10 | 0.7454 | 0.0745    | 0.1355  |
| ALL_OCCUPATIONS     | cosine         | False | 10 | 0.738  | 0.0738    | 0.1342  |
| ALL_OCCUPATIONS     | l2             | False | 10 | 0.738  | 0.0738    | 0.1342  |
| ALL_OCCUPATIONS     | ip             | False | 10 | 0.738  | 0.0738    | 0.1342  |
| LABEL_AND_DESCRIPTION | cosine       | False | 10 | 0.7196 | 0.072     | 0.1308  |
| LABEL_AND_DESCRIPTION | l2           | False | 10 | 0.7196 | 0.072     | 0.1308  |
| LABEL_AND_DESCRIPTION | ip           | False | 10 | 0.7196 | 0.072     | 0.1308  |
| DESCRIPTION         | cosine         | False | 10 | 0.7122 | 0.0712    | 0.1295  |
| DESCRIPTION         | l2             | False | 10 | 0.7122 | 0.0712    | 0.1295  |
| DESCRIPTION         | ip             | False | 10 | 0.7122 | 0.0712    | 0.1295  |
| PREFERREDLABEL      | cosine         | True  | 10 | 0.452  | 0.0452    | 0.0822  |
| PREFERREDLABEL      | l2             | True  | 10 | 0.452  | 0.0452    | 0.0822  |
| PREFERREDLABEL      | ip             | True  | 10 | 0.452  | 0.0452    | 0.0822  |
| LABEL_AND_DESCRIPTION | ip           | True  | 10 | 0.4317 | 0.0432    | 0.0785  |
| ALL_OCCUPATIONS     | cosine         | True  | 10 | 0.4299 | 0.043     | 0.0782  |
| ALL_OCCUPATIONS     | l2             | True  | 10 | 0.4299 | 0.043     | 0.0782  |
| ALL_OCCUPATIONS     | ip             | True  | 10 | 0.4299 | 0.043     | 0.0782  |
| LABEL_AND_DESCRIPTION | cosine       | True  | 10 | 0.4299 | 0.043     | 0.0782  |
| LABEL_AND_DESCRIPTION | l2           | True  | 10 | 0.4299 | 0.043     | 0.0782  |
| DESCRIPTION         | cosine         | True  | 10 | 0.3985 | 0.0399    | 0.0725  |
| DESCRIPTION         | l2             | True  | 10 | 0.3985 | 0.0399    | 0.0725  |
| DESCRIPTION         | ip             | True  | 10 | 0.3985 | 0.0399    | 0.0725  |
| PREFERREDLABEL      | cosine         | False | 5  | 0.6624 | 0.1325    | 0.2208  |
| PREFERREDLABEL      | l2             | False | 5  | 0.6624 | 0.1325    | 0.2208  |
| PREFERREDLABEL      | ip             | False | 5  | 0.6624 | 0.1325    | 0.2208  |
| ALL_OCCUPATIONS     | cosine         | False | 5  | 0.6605 | 0.1321    | 0.2202  |
| ALL_OCCUPATIONS     | l2             | False | 5  | 0.6605 | 0.1321    | 0.2202  |
| ALL_OCCUPATIONS     | ip             | False | 5  | 0.6605 | 0.1321    | 0.2202  |
| DESCRIPTION         | cosine         | False | 5  | 0.6181 | 0.1236    | 0.206   |
| DESCRIPTION         | l2             | False | 5  | 0.6181 | 0.1236    | 0.206   |
| DESCRIPTION         | ip             | False | 5  | 0.6181 | 0.1236    | 0.206   |
| LABEL_AND_DESCRIPTION | cosine       | False | 5  | 0.6144 | 0.1229    | 0.2048  |
| LABEL_AND_DESCRIPTION | l2           | False | 5  | 0.6144 | 0.1229    | 0.2048  |
| LABEL_AND_DESCRIPTION | ip           | False | 5  | 0.6144 | 0.1229    | 0.2048  |
| PREFERREDLABEL      | cosine         | True  | 5  | 0.4483 | 0.0897    | 0.1494  |
| PREFERREDLABEL      | l2             | True  | 5  | 0.4483 | 0.0897    | 0.1494  |
| PREFERREDLABEL      | ip             | True  | 5  | 0.4483 | 0.0897    | 0.1494  |
| LABEL_AND_DESCRIPTION | ip           | True  | 5  | 0.4317 | 0.0863    | 0.1439  |
| LABEL_AND_DESCRIPTION | cosine       | True  | 5  | 0.4299 | 0.086     | 0.1433  |
| LABEL_AND_DESCRIPTION | l2           | True  | 5  | 0.4299 | 0.086     | 0.1433  |
| ALL_OCCUPATIONS     | cosine         | True  | 5  | 0.4244 | 0.0849    | 0.1415  |
| ALL_OCCUPATIONS     | l2             | True  | 5  | 0.4244 | 0.0849    | 0.1415  |
| ALL_OCCUPATIONS     | ip             | True  | 5  | 0.4244 | 0.0849    | 0.1415  |
| DESCRIPTION         | cosine         | True  | 5  | 0.3967 | 0.0793    | 0.1322  |
| DESCRIPTION         | l2             | True  | 5  | 0.3967 | 0.0793    | 0.1322  |
| DESCRIPTION         | ip             | True  | 5  | 0.3967 | 0.0793    | 0.1322  |
| PREFERREDLABEL      | cosine         | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| PREFERREDLABEL      | l2             | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| PREFERREDLABEL      | ip             | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| ALL_OCCUPATIONS     | cosine         | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| ALL_OCCUPATIONS     | l2             | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| ALL_OCCUPATIONS     | ip             | False | 3  | 0.5812 | 0.1937    | 0.2906  |
| LABEL_AND_DESCRIPTION | cosine       | False | 3  | 0.548  | 0.1827    | 0.274   |
| LABEL_AND_DESCRIPTION | l2           | False | 3  | 0.548  | 0.1827    | 0.274   |
| LABEL_AND_DESCRIPTION | ip           | False | 3  | 0.548  | 0.1827    | 0.274   |
| DESCRIPTION         | cosine         | False | 3  | 0.5424 | 0.1808    | 0.2712  |
| DESCRIPTION         | l2             | False | 3  | 0.5424 | 0.1808    | 0.2712  |
| DESCRIPTION         | ip             | False | 3  | 0.5424 | 0.1808    | 0.2712  |
| PREFERREDLABEL      | cosine         | True  | 3  | 0.4317 | 0.1439    | 0.2159  |
| PREFERREDLABEL      | l2             | True  | 3  | 0.4317 | 0.1439    | 0.2159  |
| PREFERREDLABEL      | ip             | True  | 3  | 0.4317 | 0.1439    | 0.2159  |
| LABEL_AND_DESCRIPTION | cosine       | True  | 3  | 0.4188 | 0.1396    | 0.2094  |
| LABEL_AND_DESCRIPTION | l2           | True  | 3  | 0.4188 | 0.1396    | 0.2094  |
| LABEL_AND_DESCRIPTION | ip           | True  | 3  | 0.4188 | 0.1396    | 0.2094  |
| ALL_OCCUPATIONS     | cosine         | True  | 3  | 0.417  | 0.139     | 0.2085  |
| ALL_OCCUPATIONS     | l2             | True  | 3  | 0.417  | 0.139     | 0.2085  |
| ALL_OCCUPATIONS     | ip             | True  | 3  | 0.417  | 0.139     | 0.2085  |
| DESCRIPTION         | cosine         | True  | 3  | 0.3838 | 0.1279    | 0.1919  |
| DESCRIPTION         | l2             | True  | 3  | 0.3838 | 0.1279    | 0.1919  |
| DESCRIPTION         | ip             | True  | 3  | 0.3838 | 0.1279    | 0.1919  |
| PREFERREDLABEL      | cosine         | False | 1  | 0.3801 | 0.3801    | 0.3801  |
| PREFERREDLABEL      | cosine         | True  | 1  | 0.3801 | 0.3801    | 0.3801  |
| PREFERREDLABEL      | l2             | False | 1  | 0.3801 | 0.3801    | 0.3801  |
| PREFERREDLABEL      | l2             | True  | 1  | 0.3801 | 0.3801    | 0.3801  |
| PREFERREDLABEL      | ip             | False | 1  | 0.3801 | 0.3801    | 0.3801  |
| PREFERREDLABEL      | ip             | True  | 1  | 0.3801 | 0.3801    | 0.3801  |
| LABEL_AND_DESCRIPTION | cosine       | False | 1  | 0.3727 | 0.3727    | 0.3727  |
| LABEL_AND_DESCRIPTION | cosine       | True  | 1  | 0.3727 | 0.3727    | 0.3727  |
| LABEL_AND_DESCRIPTION | l2           | False | 1  | 0.3727 | 0.3727    | 0.3727  |
| LABEL_AND_DESCRIPTION | l2           | True  | 1  | 0.3727 | 0.3727    | 0.3727  |
| LABEL_AND_DESCRIPTION | ip           | False | 1  | 0.3727 | 0.3727    | 0.3727  |
| LABEL_AND_DESCRIPTION | ip           | True  | 1  | 0.3727 | 0.3727    | 0.3727  |
| ALL_OCCUPATIONS     | cosine         | False | 1  | 0.3469 | 0.3469    | 0.3469  |
| ALL_OCCUPATIONS     | cosine         | True  | 1  | 0.3469 | 0.3469    | 0.3469  |
| ALL_OCCUPATIONS     | l2             | False | 1  | 0.3469 | 0.3469    | 0.3469  |
| ALL_OCCUPATIONS     | l2             | True  | 1  | 0.3469 | 0.3469    | 0.3469  |
| ALL_OCCUPATIONS     | ip             | False | 1  | 0.3469 | 0.3469    | 0.3469  |
| ALL_OCCUPATIONS     | ip             | True  | 1  | 0.3469 | 0.3469    | 0.3469  |
| DESCRIPTION         | cosine         | False | 1  | 0.3395 | 0.3395    | 0.3395  |
| DESCRIPTION         | cosine         | True  | 1  | 0.3395 | 0.3395    | 0.3395  |
| DESCRIPTION         | l2             | False | 1  | 0.3395 | 0.3395    | 0.3395  |
| DESCRIPTION         | l2             | True  | 1  | 0.3395 | 0.3395    | 0.3395  |
| DESCRIPTION         | ip             | False | 1  | 0.3395 | 0.3395    | 0.3395  |
| DESCRIPTION         | ip             | True  | 1  | 0.3395 | 0.3395    | 0.3395  |


The result of the evaluation are as follows:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods is PREFERREDLABEL, slightly higher than ALL_OCCUPATIONS in our k of interest (3 or 5). We are choosing PREFERREDLABEL given that it has a small margin over ALL_OCCUPATIONS and it's easier to implement as its value is already present in the dataset. We think that the information contained in such label helps the model identify the correct occupation even when they are not using the same words, as the correspondence between ESCO nodes and PREFERREDLABEL is one-to-one. On the other hand, a secondary labels can appear in multiple nodes, thus confusing the model.

### Skills

| method            | score function | MMR   | k   | recall | precision | f-score |
|-------------------|----------------|-------|-----|--------|-----------|---------|
| PREFERREDLABEL    | cosine         | False | 10  | 0.4794 | 0.0479    | 0.0872  |
| PREFERREDLABEL    | l2             | False | 10  | 0.4794 | 0.0479    | 0.0872  |
| PREFERREDLABEL    | ip             | False | 10  | 0.4794 | 0.0479    | 0.0872  |
| LABEL_AND_DESCRIPTION | cosine    | False | 10  | 0.4362 | 0.0436    | 0.0793  |
| LABEL_AND_DESCRIPTION | l2        | False | 10  | 0.4362 | 0.0436    | 0.0793  |
| LABEL_AND_DESCRIPTION | ip        | False | 10  | 0.4357 | 0.0436    | 0.0792  |
| ALL_SKILLS        | cosine         | False | 10  | 0.4267 | 0.0427    | 0.0776  |
| ALL_SKILLS        | l2             | False | 10  | 0.4267 | 0.0427    | 0.0776  |
| ALL_SKILLS        | ip             | False | 10  | 0.4267 | 0.0427    | 0.0776  |
| DESCRIPTION       | cosine         | False | 10  | 0.3343 | 0.0334    | 0.0608  |
| DESCRIPTION       | l2             | False | 10  | 0.3343 | 0.0334    | 0.0608  |
| DESCRIPTION       | ip             | False | 10  | 0.3343 | 0.0334    | 0.0608  |
| PREFERREDLABEL    | l2             | True  | 10  | 0.305  | 0.0305    | 0.0555  |
| PREFERREDLABEL    | ip             | True  | 10  | 0.305  | 0.0305    | 0.0555  |
| PREFERREDLABEL    | cosine         | True  | 10  | 0.3045 | 0.0305    | 0.0554  |
| LABEL_AND_DESCRIPTION | cosine    | True  | 10  | 0.2653 | 0.0265    | 0.0482  |
| LABEL_AND_DESCRIPTION | l2        | True  | 10  | 0.2653 | 0.0265    | 0.0482  |
| LABEL_AND_DESCRIPTION | ip        | True  | 10  | 0.2653 | 0.0265    | 0.0482  |
| ALL_SKILLS        | cosine         | True  | 10  | 0.2583 | 0.0258    | 0.047   |
| ALL_SKILLS        | l2             | True  | 10  | 0.2578 | 0.0258    | 0.0469  |
| ALL_SKILLS        | ip             | True  | 10  | 0.2578 | 0.0258    | 0.0469  |
| DESCRIPTION       | l2             | True  | 10  | 0.1873 | 0.0187    | 0.0341  |
| DESCRIPTION       | cosine         | True  | 10  | 0.1868 | 0.0187    | 0.034   |
| DESCRIPTION       | ip             | True  | 10  | 0.1868 | 0.0187    | 0.034   |
| PREFERREDLABEL    | cosine         | False | 5   | 0.3954 | 0.0791    | 0.1318  |
| PREFERREDLABEL    | l2             | False | 5   | 0.3954 | 0.0791    | 0.1318  |
| PREFERREDLABEL    | ip             | False | 5   | 0.3954 | 0.0791    | 0.1318  |
| ALL_SKILLS        | cosine         | False | 5   | 0.3517 | 0.0703    | 0.1172  |
| ALL_SKILLS        | l2             | False | 5   | 0.3517 | 0.0703    | 0.1172  |
| ALL_SKILLS        | ip             | False | 5   | 0.3517 | 0.0703    | 0.1172  |
| LABEL_AND_DESCRIPTION | cosine    | False | 5   | 0.3462 | 0.0692    | 0.1154  |
| LABEL_AND_DESCRIPTION | l2        | False | 5   | 0.3462 | 0.0692    | 0.1154  |
| LABEL_AND_DESCRIPTION | ip        | False | 5   | 0.3462 | 0.0692    | 0.1154  |
| PREFERREDLABEL    | l2             | True  | 5   | 0.2911 | 0.0582    | 0.097   |
| PREFERREDLABEL    | ip             | True  | 5   | 0.2911 | 0.0582    | 0.097   |
| PREFERREDLABEL    | cosine         | True  | 5   | 0.2906 | 0.0581    | 0.0969  |
| DESCRIPTION       | cosine         | False | 5   | 0.2543 | 0.0509    | 0.0848  |
| DESCRIPTION       | l2             | False | 5   | 0.2543 | 0.0509    | 0.0848  |
| DESCRIPTION       | ip             | False | 5   | 0.2543 | 0.0509    | 0.0848  |
| LABEL_AND_DESCRIPTION | ip        | True  | 5   | 0.2529 | 0.0506    | 0.0843  |
| LABEL_AND_DESCRIPTION | cosine    | True  | 5   | 0.2524 | 0.0505    | 0.0841  |
| LABEL_AND_DESCRIPTION | l2        | True  | 5   | 0.2524 | 0.0505    | 0.0841  |
| ALL_SKILLS        | cosine         | True  | 5   | 0.2474 | 0.0495    | 0.0825  |
| ALL_SKILLS        | l2             | True  | 5   | 0.2469 | 0.0494    | 0.0823  |
| ALL_SKILLS        | ip             | True  | 5   | 0.2469 | 0.0494    | 0.0823  |
| DESCRIPTION       | cosine         | True  | 5   | 0.1764 | 0.0353    | 0.0588  |
| DESCRIPTION       | l2             | True  | 5   | 0.1764 | 0.0353    | 0.0588  |
| DESCRIPTION       | ip             | True  | 5   | 0.1759 | 0.0352    | 0.0586  |
| PREFERREDLABEL    | cosine         | False | 3   | 0.3318 | 0.1106    | 0.1659  |
| PREFERREDLABEL    | l2             | False | 3   | 0.3318 | 0.1106    | 0.1659  |
| PREFERREDLABEL    | ip             | False | 3   | 0.3318 | 0.1106    | 0.1659  |
| ALL_SKILLS        | cosine         | False | 3   | 0.2966 | 0.0989    | 0.1483  |
| ALL_SKILLS        | l2             | False | 3   | 0.2966 | 0.0989    | 0.1483  |
| ALL_SKILLS        | ip             | False | 3   | 0.2966 | 0.0989    | 0.1483  |
| LABEL_AND_DESCRIPTION | cosine    | False | 3   | 0.2881 | 0.096     | 0.1441  |
| LABEL_AND_DESCRIPTION | l2        | False | 3   | 0.2881 | 0.096     | 0.1441  |
| LABEL_AND_DESCRIPTION | ip        | False | 3   | 0.2881 | 0.096     | 0.1441  |
| PREFERREDLABEL    | l2             | True  | 3   | 0.2772 | 0.0924    | 0.1386  |
| PREFERREDLABEL    | ip             | True  | 3   | 0.2772 | 0.0924    | 0.1386  |
| PREFERREDLABEL    | cosine         | True  | 3   | 0.2762 | 0.0921    | 0.1381  |
| LABEL_AND_DESCRIPTION | ip        | True  | 3   | 0.235  | 0.0783    | 0.1175  |
| LABEL_AND_DESCRIPTION | cosine    | True  | 3   | 0.2345 | 0.0782    | 0.1172  |
| LABEL_AND_DESCRIPTION | l2        | True  | 3   | 0.2345 | 0.0782    | 0.1172  |
| ALL_SKILLS        | cosine         | True  | 3   | 0.233  | 0.0777    | 0.1165  |
| ALL_SKILLS        | l2             | True  | 3   | 0.2325 | 0.0775    | 0.1162  |
| ALL_SKILLS        | ip             | True  | 3   | 0.2325 | 0.0775    | 0.1162  |
| DESCRIPTION       | cosine         | False | 3   | 0.2067 | 0.0689    | 0.1033  |
| DESCRIPTION       | l2             | False | 3   | 0.2067 | 0.0689    | 0.1033  |
| DESCRIPTION       | ip             | False | 3   | 0.2067 | 0.0689    | 0.1033  |
| DESCRIPTION       | cosine         | True  | 3   | 0.1629 | 0.0543    | 0.0815  |
| DESCRIPTION       | l2             | True  | 3   | 0.1629 | 0.0543    | 0.0815  |
| DESCRIPTION       | ip             | True  | 3   | 0.1624 | 0.0541    | 0.0812  |
| PREFERREDLABEL    | cosine         | False | 1   | 0.2017 | 0.2017    | 0.2017  |
| PREFERREDLABEL    | cosine         | True  | 1   | 0.2017 | 0.2017    | 0.2017  |
| PREFERREDLABEL    | l2             | False | 1   | 0.2017 | 0.2017    | 0.2017  |
| PREFERREDLABEL    | l2             | True  | 1   | 0.2017 | 0.2017    | 0.2017  |
| PREFERREDLABEL    | ip             | False | 1   | 0.2017 | 0.2017    | 0.2017  |
| PREFERREDLABEL    | ip             | True  | 1   | 0.2017 | 0.2017    | 0.2017  |
| LABEL_AND_DESCRIPTION | cosine    | False | 1   | 0.1659 | 0.1659    | 0.1659  |
| LABEL_AND_DESCRIPTION | cosine    | True  | 1   | 0.1659 | 0.1659    | 0.1659  |
| LABEL_AND_DESCRIPTION | l2        | False | 1   | 0.1659 | 0.1659    | 0.1659  |
| LABEL_AND_DESCRIPTION | l2        | True  | 1   | 0.1659 | 0.1659    | 0.1659  |
| LABEL_AND_DESCRIPTION | ip        | False | 1   | 0.1659 | 0.1659    | 0.1659  |
| LABEL_AND_DESCRIPTION | ip        | True  | 1   | 0.1659 | 0.1659    | 0.1659  |
| ALL_SKILLS        | cosine         | False | 1   | 0.1595 | 0.1595    | 0.1595  |
| ALL_SKILLS        | cosine         | True  | 1   | 0.1595 | 0.1595    | 0.1595  |
| ALL_SKILLS        | l2             | False | 1   | 0.1595 | 0.1595    | 0.1595  |
| ALL_SKILLS        | l2             | True  | 1   | 0.1595 | 0.1595    | 0.1595  |
| ALL_SKILLS        | ip             | False | 1   | 0.1595 | 0.1595    | 0.1595  |
| ALL_SKILLS        | ip             | True  | 1   | 0.1595 | 0.1595    | 0.1595  |
| DESCRIPTION       | cosine         | False | 1   | 0.1172 | 0.1172    | 0.1172  |
| DESCRIPTION       | cosine         | True  | 1   | 0.1172 | 0.1172    | 0.1172  |
| DESCRIPTION       | l2             | False | 1   | 0.1172 | 0.1172    | 0.1172  |
| DESCRIPTION       | l2             | True  | 1   | 0.1172 | 0.1172    | 0.1172  |
| DESCRIPTION       | ip             | False | 1   | 0.1172 | 0.1172    | 0.1172  |
| DESCRIPTION       | ip             | True  | 1   | 0.1172 | 0.1172    | 0.1172  |


The result of the evaluation are similar to those for the occupations, that is:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods is PREFERREDLABEL, higher than ALL_SKILLS in our k of interest (3 or 5). PREFERREDLABEL has a high margin over the second best and it's easier to implement as its value is already present in the dataset. We think that the information contained in such label helps the model identify the correct skill even when they are not using the same words, as the correspondence between ESCO nodes and PREFERREDLABEL is one-to-one. On the other hand, a secondary labels can appear in multiple nodes, thus confusing the model.

In summary, for both skills and occupations we choose the hyperparameters to be without MMR, using the node embedding corresponding to preferred label and we choose the default retrieval function.

### 2. Linking skills to occupation statements

Another line of research involves understanding how to best link skills to a statement about the user's occupation. In practice, when we're only interested in understanding which skills are pertinent to the user, should we link the occupation statement directly to the skills dataset or should we focus on finding the right occupation and retrieving the related skills directly from the ESCO model?

In the evaluation that follows, we use the Occupation dataset having as ground truth the essential skills related to each occupation and we compare the two methods. The first method is as follows:

1. We fix the method and score functions as the optimal parameters for the previous search and load the corresponding **skills** collection.
2. For each element in the **occupation** test set, we find the top 100 **skills** in the collection ordered by scoring rank.
3. We filter those **skills** by maximal marginal relevance to find the top 10 **skills** with this function.
4. We evaluate the precision, recall and F-score on the top k for k=1,3,5,10 for the retrieved **skills**, using as ground truth the **essential skills** for the original **occupation**.
5. We evaluate the precision, recall and F-score on the top k for k=1,3,5,10 for the **skills** filtered by maximal marginal relevance, using as ground truth the **essential skills** for the original occupation.
6. We save the results in a dataframe to be analyzed.

On the other hand, the second method works as follows:

1. We fix the method and score functions as the optimal parameters for the previous search and load the corresponding **occupation** collection.
2. For each element in the test set, we find the top 100 **occupation** documents in the collection ordered by scoring rank.
3. We filter those documents by maximal marginal relevance to find the top 10 **occupations** with this function.
4. For each element of the test set, we consider the true value to be the set of all **essential skills** related to the occupation.
5. For each k=1,3,5,10, we consider as predicted elements the set of all **essential skills** related to the top k retrieved occupations (either in regular or MMR vector search).
5. We evaluate the precision, recall and F-score on the retrieved skills, both with regular and MMR vector search.
6. We save the results in a dataframe to be analyzed for later use.

Notice that the second method is expected to return a recall that is strictly better than the occupation search evaluation, as a correct occupation implies that all the necessary skills are retrieved as well.

In [12]:
# Create a dataset that maps occupation ESCO IDs to the corresponding Tabiya UUID of essential skills
# This allows us to evaluate how to retrieve skills from linking to occupations
occupation_id_to_esco_code = {row["ID"]:row["CODE"] for _, row in df_occupation_database.iterrows()}
skill_id_to_uuid = {row["ID"]: row["CODE"] for _, row in df_skill_database.iterrows()}
grouped_df = df_occupation_to_skills.groupby(["OCCUPATIONID","RELATIONTYPE"])["SKILLID"].agg(list).reset_index()
esco_id_to_skills_essential = {occupation_id_to_esco_code[row["OCCUPATIONID"]]:[skill_id_to_uuid[skill_id] for skill_id in row["SKILLID"]] for _, row in grouped_df.iterrows() if row["RELATIONTYPE"]=="essential"}
for occ_id, esco_code in occupation_id_to_esco_code.items():
    if esco_code not in esco_id_to_skills_essential:
        esco_id_to_skills_essential[esco_code] = []


In [13]:
from typing import Dict

def run_skill_occupation_eval(method_list: List[str], score_function_list: List[str], df_test: pd.DataFrame, esco_id_to_skills_essential: Dict[str,List[str]], embedding_column: str = "embeddings") -> pd.DataFrame:
    """Returns the results of an evaluation for occupation-related skills
    on a list of collections

    Args:
        method_list (List[str]): list of methods of interest
            to test.
        score_function_list (List[str]): list of score functions of interest
            to test.
        df_test (pd.DataFrame): test dataframe, containing an embedding column
            and a test_target column.
        esco_id_to_skills_essential (Dict[str,List[str]]): dictionary mapping each
            occupation ESCO id to a list of essential skills Tabiya UUID.

    Returns:
        pd.DataFrame: dataframe with the result of the evaluation depending on the
            different hyperparameters.
    """
    eval_data = []
    for method in method_list:
        for score in score_function_list:
            collection_name = f"occupations_{method}_{score}"
            # Fetch collection
            collection = client.get_collection(name=collection_name)
            # Initialize lists to save results
            vector_search_results, mmr_vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection)
            # Evaluate accuracy at k for k=1, 3, 5, 10
            for k in [1, 3, 5, 10]:
                # Link the retrieved ESCO codes to their essential skills
                skill_related_vs_results = [set(sum([esco_id_to_skills_essential[code] for code in elem[:k]], start=[])) for elem in vector_search_results]
                skill_related_mmr_vs_results = [set(sum([esco_id_to_skills_essential[code] for code in elem[:k]], start=[])) for elem in mmr_vector_search_results]
                # Finds precision, recall and F-score for the skills retrieved from the top k occupations
                rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(skill_related_vs_results, list(df_test["skills_essential"]))
                eval_data.append({"method":method, "score function":score, "MMR": False, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
                rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(skill_related_mmr_vs_results, list(df_test["skills_essential"]))
                eval_data.append({"method":method, "score function":score, "MMR": True, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
    # Save the results in a dataframe
    eval_df = pd.DataFrame(eval_data)
    return eval_df

In [23]:
# Initialize a folder to save the evaluation results locally
OUTPUT_EVAL_PATH = os.environ.get("OUTPUT_EVAL_PATH", "")
if OUTPUT_EVAL_PATH:
    os.makedirs(OUTPUT_EVAL_PATH, exist_ok=True)

# Evaluation of skill linking to occupation
df_skill_occupation_eval1 = run_eval("skills", ["PREFERREDLABEL"], ["cosine"], df_occupation_test, target_column="skills_essential")
if OUTPUT_EVAL_PATH:
    df_skill_occupation_eval1.to_csv(os.path.join(OUTPUT_EVAL_PATH, "occupation_skills_eval1.csv"))

df_skill_occupation_eval2 = run_skill_occupation_eval(["PREFERREDLABEL"], ["cosine"], df_occupation_test, esco_id_to_skills_essential)
if OUTPUT_EVAL_PATH:
    df_skill_occupation_eval2.to_csv(os.path.join(OUTPUT_EVAL_PATH, "occupation_skills_eval2.csv"))


### Occupation-related Skills

The first method has the following results:

| method        | k   | recall | precision | f-score |
|---------------|-----|--------|-----------|---------|
| PREFERREDLABEL| 10  | 0.0571 | 0.1413    | 0.0813  |
| PREFERREDLABEL| 5   | 0.0334 | 0.1631    | 0.0554  |
| PREFERREDLABEL| 3   | 0.0209 | 0.1697    | 0.0372  |
| PREFERREDLABEL| 1   | 0.0085 | 0.1845    | 0.0162  |

While the second method returns the following scores:

| method        | k   | recall | precision | f-score |
|---------------|-----|--------|-----------|---------|
| PREFERREDLABEL| 10  | 0.8559 | 0.1614    | 0.2716  |
| PREFERREDLABEL| 5   | 0.7772 | 0.2471    | 0.375   |
| PREFERREDLABEL| 3   | 0.7042 | 0.326     | 0.4457  |
| PREFERREDLABEL| 1   | 0.4982 | 0.5002    | 0.4992  |

We clearly retrieve much better results in the second case. In fact, there is an average of 27 essential skills for each occupation and the first experiment only retrieves a small number of them for k=1,3,5,10. Rather than retrieving large number of skills, we instead retrieve a small number of occupations in the second experiment and link them to their essential skills using the ESCO database. In this case, the recall results are much higher than in occupation linking. This is expected, since correct occupations imply correct related essential skills. However, the precision tends to decrease fast, as the number of skills increases by an average of 27k. Therefore, we suggest to keep a low value of k if we want to apply a similar process for our model.

## 3. Are job title queries better indicators than their descriptions?

In this last experiment, we want to understand whether, from the point of view of the user, returning exclusively the correct job title when asked about the past experiences is more convenient than giving a full description of the occupation that they had. The queries we generated synthetically return a randomized selection based on submitted title and job descriptions, so that not all the answers to the query highlight the specific job title of interest. We proceed to the evaluation in the following way:

1. We consider whether having the job title as query performs better than not declaring it explicitly.
2. We consider the subset of the test set in which the job title doesn't perform well and we evaluate if other methods that mention the description in the target node perform better than PREFERREDLABEL.

In the first case, we compare the best result for each k to the corresponding best results from the previous evaluations.

In [10]:
# Embedding the title queries in the occupation test set
title_test_occupation_embeddings = embed_strings_in_batch(list(df_occupation_test["title"]), model)
df_occupation_test["title_embeddings"] = title_test_occupation_embeddings


In [15]:
# Evaluation of occupation
df_occupation_eval = run_eval("occupations", method_list =list(function_to_occupation_method.keys()), score_function_list=["cosine"], df_test=df_occupation_test, embedding_column="title_embeddings")
if OUTPUT_EVAL_PATH:
    df_occupation_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "title_occupation_eval.csv"))

## Occupation search

| Method              | k   | Recall | Precision | F-score | Input Type                |
|---------------------|-----|--------|-----------|---------|---------------------------|
| ALL_OCCUPATIONS     | 10  | 0.7491 | 0.0749    | 0.1362  | Title                     |
| ALL_OCCUPATIONS     | 10  | 0.738  | 0.0738    | 0.1342  | Synthetic Query           |
| PREFERREDLABEL      | 10  | 0.7362 | 0.0736    | 0.1338  | Title                     |
| PREFERREDLABEL      | 10  | 0.7454 | 0.0745    | 0.1355  | Synthetic Query           |
| DESCRIPTION         | 10  | 0.7196 | 0.072     | 0.1308  | Title                     |
| DESCRIPTION         | 10  | 0.7122 | 0.0712    | 0.1295  | Synthetic Query           |
| LABEL_AND_DESCRIPTION | 10 | 0.6956 | 0.0696    | 0.1265  | Title                     |
| LABEL_AND_DESCRIPTION | 10 | 0.7196 | 0.072     | 0.1308  | Synthetic Query           |
| ALL_OCCUPATIONS     | 5   | 0.6882 | 0.1376    | 0.2294  | Title                     |
| ALL_OCCUPATIONS     | 5   | 0.6605 | 0.1321    | 0.2202  | Synthetic Query           |
| PREFERREDLABEL      | 5   | 0.6716 | 0.1343    | 0.2239  | Title                     |
| PREFERREDLABEL      | 5   | 0.6624 | 0.1325    | 0.2208  | Synthetic Query           |
| DESCRIPTION         | 5   | 0.6458 | 0.1292    | 0.2153  | Title                     |
| DESCRIPTION         | 5   | 0.6181 | 0.1236    | 0.206   | Synthetic Query           |
| LABEL_AND_DESCRIPTION | 5 | 0.6347 | 0.1269    | 0.2116  | Title                     |
| LABEL_AND_DESCRIPTION | 5 | 0.6144 | 0.1229    | 0.2048  | Synthetic Query           |
| PREFERREDLABEL      | 3   | 0.6236 | 0.2079    | 0.3118  | Title                     |
| PREFERREDLABEL      | 3   | 0.5812 | 0.1937    | 0.2906  | Synthetic Query           |
| ALL_OCCUPATIONS     | 3   | 0.6162 | 0.2054    | 0.3081  | Title                     |
| ALL_OCCUPATIONS     | 3   | 0.5812 | 0.1937    | 0.2906  | Synthetic Query           |
| DESCRIPTION         | 3   | 0.5775 | 0.1925    | 0.2887  | Title                     |
| DESCRIPTION         | 3   | 0.5424 | 0.1808    | 0.2712  | Synthetic Query           |
| LABEL_AND_DESCRIPTION | 3 | 0.559  | 0.1863    | 0.2795  | Title                     |
| LABEL_AND_DESCRIPTION | 3 | 0.548  | 0.1827    | 0.274   | Synthetic Query           |
| PREFERREDLABEL      | 1   | 0.4354 | 0.4354    | 0.4354  | Title                     |
| PREFERREDLABEL      | 1   | 0.3801 | 0.3801    | 0.3801  | Synthetic Query           |
| ALL_OCCUPATIONS     | 1   | 0.4262 | 0.4262    | 0.4262  | Title                     |
| ALL_OCCUPATIONS     | 1   | 0.3469 | 0.3469    | 0.3469  | Synthetic Query           |
| LABEL_AND_DESCRIPTION | 1 | 0.4133 | 0.4133    | 0.4133  | Title                     |
| LABEL_AND_DESCRIPTION | 1 | 0.3727 | 0.3727    | 0.3727  | Synthetic Query           |
| DESCRIPTION         | 1   | 0.393  | 0.393     | 0.393   | Title                     |
| DESCRIPTION         | 1   | 0.3395 | 0.3395    | 0.3395  | Synthetic Query           |


When comparing these results to those in the first evaluation, we notice a couple of things:
1. For lower values of k using the title as query with the **preferred label** as target returns a higher recall than using the synthetic query. For the preferred label target, this advantage tends to reduce as k grows and is actually negative for k=10. This happens because the synthetic query might contain more information that can be linked to the preferred label when the title is not found within the first 3 or 5.
2. For higher values of k, using the title as query with the **all occupations** method, which includes secondary labels and description performs better than using the synthetic query. This happens because when the title is not identified with the preferred label, there is a chance it could match a secondary label that is present in the all occupations method. 
3. The overall title performance is better than the synthetic query, although this gap is smaller for k=10. This implies that there is an inherent advantage in using models of Named Entity Recognition or asking the user directly for the title of his previous jobs.

We now ask what is the nature of those datapoints whose elements are not retrieved through the job title when k=1. Can they be identified with other fields of the collection? For that reason, we run an evaluation both on their title and on their synthetic query and compare the results.

In [33]:
# Find all the results of linking the title to the preferred label and consider the filtered dataframe in which 
# this result doesn't coincide with the ground truth
label_collection = client.get_collection("occupations_PREFERREDLABEL_cosine")
df_occupation_test["title_results_preferredlabel"] = df_occupation_test["title_embeddings"].apply(lambda x: get_results_from_embeddings([x], label_collection)[0][0][0])

filtered_df = df_occupation_test[(df_occupation_test["esco_code"]!=df_occupation_test["title_results_preferredlabel"])]

In [34]:
df_eval = run_eval("occupations", method_list =list(function_to_occupation_method.keys()), score_function_list=["cosine"], df_test=filtered_df, embedding_column="embeddings")
if OUTPUT_EVAL_PATH:
    df_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "description_eval.csv"))

df_eval = run_eval("occupations", method_list =["DESCRIPTION", "ALL_OCCUPATIONS", "LABEL_AND_DESCRIPTION"], score_function_list=["cosine"], df_test=false_title_df, embedding_column="title_embeddings")
if OUTPUT_EVAL_PATH:
    df_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "title_description_eval.csv"))

We restrict the evaluation results to the case k=1 and observe how much additional information can be given by the other fields of the collection.

### Title linking to fields

| method               | score function | k   | recall | precision | f-score |
|----------------------|----------------|-----|--------|-----------|---------|
| ALL_OCCUPATIONS     | cosine         | 1   | 0.1438 | 0.1438    | 0.1438  |
| DESCRIPTION          | cosine         | 1   | 0.1046 | 0.1046    | 0.1046  |
| LABEL_AND_DESCRIPTION| cosine         | 1   | 0.098  | 0.098     | 0.098   |

### Synthetic query linking to fields

| method               | score function | k   | recall | precision | f-score |
|----------------------|----------------|-----|--------|-----------|---------|
| ALL_OCCUPATIONS     | cosine         | 1   | 0.1895 | 0.1895    | 0.1895  |
| LABEL_AND_DESCRIPTION| cosine         | 1   | 0.1895 | 0.1895    | 0.1895  |
| DESCRIPTION          | cosine         | 1   | 0.183  | 0.183     | 0.183   |
| PREFERREDLABEL       | cosine         | 1   | 0.1699 | 0.1699    | 0.1699  |

Interestingly the largest recall boost is given by linking to the ALL_OCCUPATIONS field combination, which is the only one that contains secondary labels. In case our query coincides with the title, this is probably the ground for the 5% recall boost when compared to the other fields. In the case of the synthetic query, however, we see how this is pretty similar to the description, so that most likely we gain from the similarity between the description of the experience and the description field in the node. 

We suggest that if an NER method were to be implemented before the linking, we could experiment a composite method which links the NER-retrieved span in a sentence to the PREFERREDLABEL, but any whole sentence with no retrieved span to the ALL_OCCUPATIONS field. This is not guaranteed to benefit from the gains of this past analysis, as it is strongly dependent on the NER performance, but might benefit from it if we knew that the job titles for which we are confident in the NER are also those that can be easily linked to ESCO.

## Summary

The purpose of our study has been to evaluate linking models to the Occupation and Skill ESCO taxonomies. We chose to do so by generating synthetic queries based on a dataset of 542 job descriptions with the corresponding ESCO code for the occupations and of 1054 sentences containing 2013 skills with the corresponding Tabiya UUID. Those synthetic queries are assumed to be similar to the input that an hypothetical user of the Compass platform would submit. We also fixed the embedding model to be VertexAI's Gecko003.

We first focused the evaluation on a selection of hyperparameters that would guarantee the maximum recall at various values of retrieved nodes k. We assumed that k would be decided in advance and that we would prioritize retrieving the ground truth within the first k documents (recall-based approach) rather than having most retrieved documents be the ground truth (precision-based approach). This would be the case because we could add a validation step in which the user would confirm which skills would be relevant to their experience. The hyperparameter we considered were 
* The score function (euclidean distance, scalar product or cosine similarity);
* The usage of Maximal Marginal Relevance (MMR) or not to choose the top documents;
* How to embed each node as a combination of the collection fields.

We found that both for the occupations and for the skills, it made no difference which score function should be used. Moreover, we found that most correct nodes could be found within the first few documents, so that using MMR would not be beneficial to our purpose. Finally we discovered that using only the preferred label guarantees a higher recall at all values of k. This happens most likely because the label encodes the core meaning of the job, even when it's not referred to directly and because it is less ambiguous than including the description or the secondary label.

The second evaluation involved understanding how to retrieve skills that are connected to a given occupation. A first naive approach involved linking the occupation synthetic queries to the skill database and calculating the recall using the essential skills related to the ground truth occupation as true values. This however, was not very efficient for our values of k as each occupation has an average of 27 related essential skills. Of the top 10 skills, less than 2 on average were included in these essential skills, so we decided to change approach.

The approach that yielded better results involved linking occupations first and then considering as predictions all the essential skills related to the occupations, compared to the ground truth of all the essential skills of the true occupation. It is clear that the recall of this method would be higher than the recall of the occupation linking evaluation, as a correctly retrieved occupation would guarantee that all its essential skills are correctly retrieved. However, the interesting component is seeing how wrongly retrieved occupations might have skills in common with the correct one, thus improving the overall recall of skills. This is indeed already the case for k=1, so that about half the essential skills are retrieved on average. For higher values of k, this leads to very large recall values at the expenses of the precision. We included the F-score to make sure we could choose the k that would maximizes this trade-off, which is a value between 1 and 3. The method could be further improved by adding a ranking system tha would present the most relevant skills for measures given outside of this experiment (such as relevance, transferability and so on).

Finally, we were interested in understanding whether we would benefit from a Named Entity Recognition (NER) model that would select subspans of the query to be linked to the occupation. We did so by linking only the titles and comparing their results to the ground truth, both for the occupation evaluation and for the essential skills of the second experiment. We found that indeed linking the title guarantees an increment in recall for low values of k, but that this gains tend to disappear for higher values of k. This suggests that we might benefit from a NER model if we decide to retrieve a lower number of elements. We also analyzed the residual elements in which the title linking failed and found out that linking the whole sentence to the description was more successful than linking it to the preferred label. We suggested for this reason that if a NER model were to be implemented, we could consider linking the retrieved subspan to the preferred label and the sentences in which no span is retrieved to a combination of all the fields.

Further work will consist of understanding how to best rank the retrieved skills.

In [ ]:
# Best results are given without MMR and using only the preferred label for inputs
def evaluate(model: TextEmbeddingModel, database: pd.DataFrame, test_set: pd.DataFrame, target_column: str, k: int) -> Tuple[float, float]:
    """Returns the precision and recall at k for the vector search through the embedding generated 
    by a given model on a provided dataset. The model must be a TextEmbeddingModel,
    the database must be a Dataframe containing the columns 'PREFERREDLABEL' and 'CODE',
    while the test set must be a Dataframe containing the columns 'synthetic_query' and
    target_column.

    Args:
        model (TextEmbeddingModel): model to be evaluated.
        database (pd.DataFrame): database from which the data should be retrieved.
        test_set (pd.DataFrame): test set for the evaluation.
        target_column (str): name of the target column with the true positives.
            The content should be a list of strings.
        k (int): k for the metrics at k to be evaluated.

    Returns:
        Tuple[float, float, float]: precision, recall and f-score at k
    """
    assert k<=100
    # Check that the elements of the target column are lists
    assert all(test_set[target_column].apply(lambda x: isinstance(x, list)))
    # Calculate embeddings for the database
    db_embeddings = embed_strings_in_batch(list(database['PREFERREDLABEL']), model)
    # Calculate embeddings for the test set
    test_embeddings = embed_strings_in_batch(list(test_set['synthetic_query']), model)
    # Load a collection from chromadb and saves the data
    client = chromadb.Client()
    collection = client.create_collection(name='eval')
    collection.add(
        documents = list(database['PREFERREDLABEL']),
        metadatas = [{"CODE": row[target_column]} for _, row in database.iterrows()],
        embeddings = db_embeddings,
        ids = [f"id_{i}" for i in range(len(database))]
    )
    # Save the data retrieved from the test set
    vector_search_results = []
    for test_embedding in test_embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    # Evaluate precision, recall and f-score at k
    prec = precision_at_k(vector_search_results, list(test_set[target_column]), k)
    rec = recall_at_k(vector_search_results, list(test_set[target_column]), k)
    return prec, rec, f_score(prec, rec)
